# Notebook 3: Cross-Encoder Re-Ranking
**Improve retrieval precision before sending context to the LLM**

## The problem

Bi-encoder search (k-NN with `all-MiniLM-L6-v2`) encodes query and document **independently**.
It's fast, but it can't model fine-grained interactions between query and passage tokens.

## The solution: two-stage retrieval

```md
Query
  │
  ▼
[Bi-encoder k-NN]    ← fast, retrieve top-20 candidates from full index
  │
  ▼
[Cross-encoder]      ← slow but accurate, re-score 20 → keep top-5
  │
  ▼
[Ollama RAG]         ← generate answer from the 5 best chunks
```

A **cross-encoder** processes `(query, passage)` pairs jointly through a transformer,
producing much more accurate relevance scores, but at O(N) cost per query,
so we only apply it to the top-k candidates.

### Prerequisites
- Notebooks 01 + 02 completed (podcasts indexed, RAG working)

## 0 · Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from sentence_transformers import SentenceTransformer
from src.opensearch_client import get_client, knn_search, hybrid_search, INDEX_NAME
from src.reranker import rerank, get_reranker
from src.rag import ask, build_context

client = get_client()
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')

def embed(text: str) -> list[float]:
    return bi_encoder.encode(text, normalize_embeddings=True).tolist()

count = client.count(index=INDEX_NAME)['count']
print(f'✅ {count} chunks indexed')

✅ 1018 chunks indexed


## 1 · Load the Cross-Encoder

We use `cross-encoder/ms-marco-MiniLM-L-6-v2`. A lightweight cross-encoder
trained on the MS MARCO passage ranking dataset.

| Model | Parameters | Use case |
|---|---|---|
| `ms-marco-MiniLM-L-6-v2` | 22M | Fast re-ranking (~5ms per pair) |
| `ms-marco-MiniLM-L-12-v2` | 33M | Better accuracy, still fast |
| `ms-marco-electra-base` | 110M | Best accuracy, slower |

In [2]:
print('Loading cross-encoder (downloads ~90 MB on first run)...')
cross_encoder = get_reranker('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('✅ Cross-encoder loaded')

# Quick demo: score a single (query, passage) pair
score = cross_encoder.predict([('What is HNSW?', 'HNSW is a graph-based algorithm for approximate nearest neighbor search.')])
print(f'\nDemo relevance score: {score[0]:.4f}  (higher = more relevant)')

Loading cross-encoder (downloads ~90 MB on first run)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Cross-encoder loaded

Demo relevance score: 10.0525  (higher = more relevant)


## 2 · Retrieve → Re-rank → Compare

We'll retrieve 20 candidates with k-NN, then re-rank to top 5.
Watch how the **ordering changes**, the cross-encoder often promotes highly relevant chunks that the bi-encoder under-ranked.

In [3]:
query = "What are the challenges of deploying vector search in production at scale?"

# Stage 1: Bi-encoder retrieval (top-20)
candidates = knn_search(client, embed(query), k=20)

print(f"Stage 1: Bi-encoder top-5 (of {len(candidates)} retrieved):")
print('─' * 70)
for i, r in enumerate(candidates[:5], 1):
    print(f"  [{i}] kNN={r['score']:.4f}  {r['episode_title'][:55]}")
    print(f"       {r['chunk_text'][:120]}...")
    print()

Stage 1: Bi-encoder top-5 (of 20 retrieved):
──────────────────────────────────────────────────────────────────────
  [1] kNN=0.7101  Filip Haltmayer (Data Engineer, Ziliz) on Milvus vector
       they usually bring the vectors themselves. We're currently going working on as well as we're building something a little...

  [2] kNN=0.7006  Connor Shorten - Research Scientist, Weaviate - ChatGPT
       opportunities for vector search to appeal to to the you know search engine builder so maybe some other engines like reco...

  [3] kNN=0.6972  Greg Kogan - Pinecone - Vector Podcast with Dmitry Kan
       tech company and also have better things to focus on, especially when supporting your search and recommender systems wou...

  [4] kNN=0.6966  Atita Arora - Search Relevance Consultant - Revolutioni
       know, introduce vector database into the mix or whatever. But I think you can, right? Like with solar, Alexander Benedet...

  [5] kNN=0.6962  Yaniv Vaknin - Director of Product, Searchi

In [4]:
# Stage 2: Cross-encoder re-ranking
reranked = rerank(query, candidates, top_n=5)

print(f"Stage 2: Cross-encoder top-5 (re-ranked from {len(candidates)}):")
print('─' * 70)
for i, r in enumerate(reranked, 1):
    print(f"  [{i}] rerank={r['rerank_score']:.4f}  (was kNN={r['original_score']:.4f})  {r['episode_title'][:45]}")
    print(f"       {r['chunk_text'][:120]}...")
    print()

Stage 2: Cross-encoder top-5 (re-ranked from 20):
──────────────────────────────────────────────────────────────────────
  [1] rerank=0.1912  (was kNN=0.6753)  Yaniv Vaknin - Director of Product, Searchium
       do much right I just need to install the plugin of course I need to have credentials. And what I wanted to say is that i...

  [2] rerank=-0.4528  (was kNN=0.6972)  Greg Kogan - Pinecone - Vector Podcast with D
       tech company and also have better things to focus on, especially when supporting your search and recommender systems wou...

  [3] rerank=-1.3500  (was kNN=0.6875)  Joan Fontanals - Principal Engineer - Jina AI
       try to develop our own solutions for the use case that we may feel more worth so that is I mean the one is out there but...

  [4] rerank=-1.6545  (was kNN=0.7006)  Connor Shorten - Research Scientist, Weaviate
       opportunities for vector search to appeal to to the you know search engine builder so maybe some other engines like reco...

  [5] re

## 3 · Side-by-Side RAG Comparison

Let's see how re-ranking affects the final generated answer.

In [5]:
query = "How did Yury Malkov come up with HNSW? What was his motivation?"

# Without re-ranking
vanilla_hits = knn_search(client, embed(query), k=5)

# With re-ranking (retrieve 20, keep 5)
candidates = knn_search(client, embed(query), k=20)
reranked_hits = rerank(query, candidates, top_n=5)

print('=' * 70)
print('ANSWER WITHOUT RE-RANKING')
print('=' * 70)
print('Sources:', ', '.join(set(h['episode_title'] for h in vanilla_hits)))
print()
answer_vanilla = ask(query, vanilla_hits)
print(answer_vanilla)

print()
print('=' * 70)
print('ANSWER WITH CROSS-ENCODER RE-RANKING')
print('=' * 70)
print('Sources:', ', '.join(set(h['episode_title'] for h in reranked_hits)))
print()
answer_reranked = ask(query, reranked_hits)
print(answer_reranked)

ANSWER WITHOUT RE-RANKING
Sources: Bob van Luijt (CEO, Semi) on the Weaviate vector search engine, Debunking myths of vector search and LLMs with Leo Boytsov, Yury Malkov - Staff Engineer, Twitter - Author of the most adopted ANN algorithm HNSW

Based on the provided transcript excerpts from the episode *Debunking myths of vector search and LLMs with Leo Boytsov*, there is no information explaining how Yuri Malkov came up with HNSW or his specific motivation for creating it.

The excerpts only note that:
- Yuri shared early versions of the algorithm with Leo Boytsov.
- His contributions helped refine SW graphs and played a role in winning the NNBG Mark competition.
- His work had a significant downstream impact on later systems like FAISS and the broader vector search community.

For details on HNSW's origins or Yuri's motivation, that information is not contained in these provided transcripts.

ANSWER WITH CROSS-ENCODER RE-RANKING
Sources: Jo Bergum - Distinguished Engineer, Yahoo! Ve

## 4 · Measure Re-Ranking Impact

Let's run a batch of queries and measure how many results change position after re-ranking.

In [6]:
test_queries = [
    "What is the difference between dense and sparse vectors?",
    "How does Weaviate handle multi-tenancy?",
    "What role does quantization play in vector search?",
    "How to measure search relevance in e-commerce?",
    "What is learning to rank and how does it work?",
]

for q in test_queries:
    candidates = knn_search(client, embed(q), k=20)
    reranked = rerank(q, candidates, top_n=5)
    
    # Check how many of the top-5 bi-encoder results survive re-ranking
    original_top5_ids = {c['chunk_text'][:80] for c in candidates[:5]}
    reranked_top5_ids = {r['chunk_text'][:80] for r in reranked}
    overlap = len(original_top5_ids & reranked_top5_ids)
    
    print(f'Q: {q[:60]}...')
    print(f'   Top-5 overlap: {overlap}/5. re-ranker changed {5-overlap} results')
    if reranked:
        print(f'   Best: [{reranked[0]["rerank_score"]:.3f}] {reranked[0]["episode_title"][:50]}')
    print()

Q: What is the difference between dense and sparse vectors?...
   Top-5 overlap: 2/5. re-ranker changed 3 results
   Best: [2.901] Doug Turnbull - Staff Relevance Engineer, Shopify 

Q: How does Weaviate handle multi-tenancy?...
   Top-5 overlap: 1/5. re-ranker changed 4 results
   Best: [-0.792] Economical way of serving vector search workloads 

Q: What role does quantization play in vector search?...
   Top-5 overlap: 1/5. re-ranker changed 4 results
   Best: [-0.330] Filip Haltmayer (Data Engineer, Ziliz) on Milvus v

Q: How to measure search relevance in e-commerce?...
   Top-5 overlap: 2/5. re-ranker changed 3 results
   Best: [1.026] Grant Ingersoll - Fractional CTO, Leading Search C

Q: What is learning to rank and how does it work?...
   Top-5 overlap: 4/5. re-ranker changed 1 results
   Best: [4.614] Doug Turnbull - Staff Relevance Engineer, Shopify 



## Key Takeaways

| Aspect | Bi-encoder (k-NN) | Cross-encoder (re-ranker) |
|---|---|---|
| Speed | O(1) via HNSW graph | O(N) - scores each pair |
| Accuracy | Good | Much better |
| Use case | First-stage retrieval | Second-stage re-scoring |
| Typical k | 10–100 | Top 5–20 from stage 1 |

**Rule of thumb**: retrieve 3–4x more candidates than you need, then re-rank down.
For RAG, re-ranking directly improves answer quality by feeding the LLM the *most* relevant chunks.